In [0]:
# Configure paths

STORAGE_ACCOUNT = "storagelokiiiiiiii"
CONTAINER = "lakehouse"

ADLS_ROOT = (
    f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
)

BRONZE_PATH = f"{ADLS_ROOT}/bronze/orders/"
CHECKPOINT_PATH = f"{ADLS_ROOT}/checkpoints/orders/"

print("Bronze output:", BRONZE_PATH)
print("Checkpoint:", CHECKPOINT_PATH)

In [0]:
secret_value = dbutils.secrets.get(
    scope="eh-orders-demo",
    key="listen-orders"
).strip().strip("'\"")

if secret_value.startswith("Endpoint=sb://"):
    secret_type = "full_connection_string"
elif secret_value.startswith("SharedAccessSignature "):
    secret_type = "sas_token"
elif ";" in secret_value and "=" in secret_value:
    secret_type = "connection_string_or_other"
else:
    secret_type = "key_or_other"

print("Secret format:", secret_type)
print("Secret length:", len(secret_value))

In [0]:
# Configure the Event Hubs source

EH_NAMESPACE = "hns-orders-demo"
EH_NAME = "orders"

# Must match the policy name in Azure Event Hubs
POLICY_NAME = "listen-orders"

SHARED_KEY = dbutils.secrets.get(
    scope="eh-orders-demo",
    key="listen-orders"
).strip().strip("'\"")

KAFKA_OPTIONS = {
    "kafka.bootstrap.servers":
        f"{EH_NAMESPACE}.servicebus.windows.net:9093",

    "subscribe": EH_NAME,

    "kafka.security.protocol": "SASL_SSL",
    "kafka.sasl.mechanism": "PLAIN",

    "kafka.sasl.jaas.config": (
        "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule "
        'required username="$ConnectionString" '
        f'password="Endpoint=sb://{EH_NAMESPACE}.servicebus.windows.net/;'
        f'SharedAccessKeyName={POLICY_NAME};'
        f'SharedAccessKey={SHARED_KEY};'
        f'EntityPath={EH_NAME}";'
    )
}

print("Kafka options prepared.")

In [0]:
# Read the stream from Event Hubs and Prepare the Bronze records

raw_stream = (
    spark.readStream
        .format("kafka")
        .options(**KAFKA_OPTIONS)
        .load()
)

from pyspark.sql.functions import col, current_timestamp

bronze_stream = raw_stream.select(
    col("value").cast("string").alias("event_json"),
    col("timestamp").alias("eventhub_timestamp"),
    col("topic").alias("eventhub_topic"),
    col("partition").alias("eventhub_partition"),
    col("offset").alias("eventhub_offset"),
    current_timestamp().alias("ingest_timestamp")
)

In [0]:
# Write available events to Parquet and then stop
query = (
    bronze_stream.writeStream
        .format("parquet")
        .outputMode("append")
        .option("path", BRONZE_PATH)
        .option("checkpointLocation", CHECKPOINT_PATH)
        .trigger(availableNow=True)
        .start()
)

query.awaitTermination()

print("Bronze ingestion run completed.")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS dev.bronze.orders_raw
    USING PARQUET
    LOCATION '{BRONZE_PATH}'
""")